# Sampling and Confidence Intervals

**Notebook 2 of 5.** `01_distributions.ipynb` described **single penguins**: how the body
masses of 68 Chinstraps spread around their mean of 3,733 g. This notebook asks about that
mean itself.

Those 68 penguins are not all the Chinstraps there are. They are a **sample**, and 3,733 g
is the average of this sample only. Weigh a different 68 and you would get a different
average. So how far from the true average of all Chinstraps could 3,733 g be?

The answer is the second idea every hypothesis test is built on, the **standard error**,
and it gives you a **confidence interval**: a range of plausible values for the true
average. By the end of the notebook that range will already be answering a yes-or-no
question, which is what a test does in `03_how_a_test_works.ipynb`.

By the end you should be able to:

- tell a population parameter apart from a sample statistic.
- show by simulation how much a sample average varies, and why a bigger sample varies
  less.
- compute the standard error of a mean from a single sample.
- build a 95% confidence interval for a mean, and say what the 95% refers to.
- use a confidence interval to judge whether a claimed value is plausible.

First the libraries, then the same Chinstraps as in notebook 01.

In [ ]:
# data analysis
import numpy as np
import pandas as pd

# visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# statistics
from scipy import stats

sns.set_style("whitegrid")

In [ ]:
df_penguins = pd.read_csv("../data/penguins.csv")

# the 68 Chinstraps, and their body mass
chinstrap = df_penguins[df_penguins["species"] == "Chinstrap"]
mass = chinstrap["body_mass_g"]

## 1. Population and sample

Two words carry this whole notebook.

- The **population** is every individual you want to draw a conclusion about: *all*
  Chinstrap penguins. You almost never get to measure it.
- A **sample** is the part you actually measured: our 68.

A number that describes the population is a **parameter**. It is fixed, but unknown. The
same number computed on the sample is a **statistic**. It is known, but it would come out
differently on a different sample. Statisticians give them different letters so they are
never confused:

| | Population parameter | Sample statistic |
| :--- | :---: | :---: |
| mean | $\mu$ | $\bar{x}$ |
| standard deviation | $\sigma$ | $s$ |
| number of individuals | | $n$ |

Our sample statistics are $\bar{x} = 3{,}733$ g, $s = 384$ g and $n = 68$. The question of
this notebook is how far $\bar{x}$ could be from $\mu$.

## 2. How much does an average wobble?

We cannot go back to Antarctica for another 68 penguins, but a computer can do the
next-best thing. **Pretend we know the population**: suppose all Chinstraps really have
mean body mass 3,750 g and standard deviation 400 g. Then draw samples of 68 from it, over
and over, and see how much their averages vary.

> **Note.** The 3,750 g and 400 g are invented for the simulation, round numbers close to
> our sample. The point is that *we* chose them, so we know exactly what the right answer
> is and can see how close each sample gets to it.

In [ ]:
# a random number generator with a fixed seed, so the results are the same every run
rng = np.random.default_rng(42)

# the pretend population
mu = 3750
sigma = 400

# one sample of 68 penguins, and its average
one_sample = rng.normal(loc=mu, scale=sigma, size=68)
print(f"one sample of 68: average {one_sample.mean():.0f} g")

# 1,000 samples of 68 at once: a table with 1,000 rows, one sample per row
samples = rng.normal(loc=mu, scale=sigma, size=(1000, 68))
sample_means = samples.mean(axis=1)  # the average of each row
print(f"1,000 samples of 68: averages from {sample_means.min():.0f} g "
      f"to {sample_means.max():.0f} g")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(sample_means, bins=30, ax=ax)
ax.axvline(mu, color="crimson", linestyle="--", linewidth=2, label=f"true mean, {mu} g")
ax.set_title("The averages of 1,000 samples of 68 penguins")
ax.set_xlabel("sample average body mass (g)")
ax.legend()
plt.show()

This histogram is not of penguins. Each bar counts **sample averages**, and the shape they
make has a name: the **sampling distribution of the mean**. Three things about it matter.

1. **It is centred on the true mean.** Some samples come out heavy and some light, but on
   average they are right.
2. **It is much narrower than the penguins themselves.** Individual Chinstraps spread over
   well over a thousand grams, but averages of 68 mostly land within about a hundred grams
   of 3,750. Averaging cancels out the heavy and the light.
3. **It is bell-shaped.** That is the normal curve from notebook 01 again, which means all
   of its tools apply: areas, `ppf`, and 1.96.

The spread of this distribution, the standard deviation of the sample averages, is the
**standard error**. It measures how much an average wobbles from one sample to the next.

In [ ]:
print(f"standard error, measured from the 1,000 averages: {sample_means.std():.1f} g")

### Bigger samples wobble less

The standard error depends on the sample size. Repeat the simulation with samples of 10,
68 and 300 penguins:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True)

for ax, size in zip(axes, [10, 68, 300]):
    # 1,000 samples of this size, and the average of each
    means = rng.normal(loc=mu, scale=sigma, size=(1000, size)).mean(axis=1)

    sns.histplot(means, bins=30, ax=ax)
    ax.axvline(mu, color="crimson", linestyle="--", linewidth=2)
    ax.set_title(f"n = {size}: standard error {means.std():.0f} g")
    ax.set_xlabel("sample average (g)")

plt.tight_layout()
plt.show()

The bigger the sample, the tighter the averages. The relationship is exact, and it gives
the formula you will use from now on:

$$
\text{standard error} = \frac{\sigma}{\sqrt{n}}
$$

For $n = 68$ that is $400 / \sqrt{68} \approx 48.5$ g, close to the 50 g the 1,000
samples measured. They do not match exactly because 1,000 samples are themselves only a
sample of all the samples there could be. The square root is why precision is expensive: **four times the data only halves
the standard error.**

### Even when the population is not bell-shaped

In section 4 of notebook 01, the body mass of all penguins pooled together was lopsided,
with a second hump, and a normal curve fitted it badly. Now take averages of it instead.

In [ ]:
# every penguin with a recorded body mass, all species together
all_mass = df_penguins["body_mass_g"].dropna()

# 1,000 samples of 68 drawn from those 342 penguins, and the average of each
pooled_means = rng.choice(all_mass, size=(1000, 68)).mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(all_mass, bins=25, ax=axes[0])
axes[0].set_title("Individual penguins: lopsided")
axes[0].set_xlabel("body mass (g)")

sns.histplot(pooled_means, bins=30, ax=axes[1])
axes[1].set_title("Averages of 68 penguins: bell-shaped")
axes[1].set_xlabel("sample average body mass (g)")

plt.tight_layout()
plt.show()

The individuals are lopsided; their averages are not. This is the **central limit
theorem**: for a large enough sample, averages are close to normally distributed whatever
the shape of the data they come from. A common rule of thumb is 30 or more.

That is why tests about averages work so widely. It is also the reason, in notebook 04, a
t-test can still be trusted on skewed restaurant bills.

## 3. The standard error from a single sample

In real life there is only one sample and the population $\sigma$ is unknown. So we
estimate it with the sample's own standard deviation, $s$:

$$
SE = \frac{s}{\sqrt{n}}
$$

That one line tells you, from 68 penguins, how much their average would wobble if you could
repeat the whole expedition.

In [ ]:
n = len(mass)
mean_mass = mass.mean()
std_mass = mass.std()

# the standard error of the mean, estimated from this one sample
se_mass = std_mass / np.sqrt(n)

print(f"sample average   : {mean_mass:.1f} g")
print(f"standard error   : {se_mass:.1f} g")

Our average of 3,733 g comes with a standard error of about 47 g. Keep both numbers: the
next section turns them into a range, and notebook 03 turns them into a test.

## 4. A 95% confidence interval

The sampling distribution is a normal curve, and notebook 01 showed that the middle 95 per
cent of any normal curve lies within 1.96 standard deviations of its centre. The standard
deviation of *this* curve is the standard error. So:

$$
\bar{x} \pm 1.96 \times SE
$$

is a range that catches the true mean for 95 per cent of samples. That range is a **95%
confidence interval**.

In [ ]:
# the normal version: the sample average plus and minus 1.96 standard errors
z_crit = stats.norm.ppf(0.975)
print(f"normal: {mean_mass - z_crit * se_mass:.0f} g to {mean_mass + z_crit * se_mass:.0f} g")

### One correction: the t-distribution

There is a catch. We did not know $\sigma$, so we estimated it with $s$, and that estimate
has its own wobble. The **t-distribution** from section 5 of notebook 01 accounts for it. As
you saw there, it looks like the normal curve with slightly heavier tails, and how heavy depends on the **degrees of freedom**, here
$n - 1 = 67$. The fewer penguins, the heavier the tails, and the wider the interval must be
to stay honest.

In [ ]:
x = np.linspace(-4, 4, 400)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, stats.norm.pdf(x), color="black", linewidth=2, label="normal")
ax.plot(x, stats.t.pdf(x, df=4), color="crimson", linestyle="--", label="t, 4 degrees of freedom")
ax.plot(x, stats.t.pdf(x, df=67), color="steelblue", linestyle=":", linewidth=2,
        label="t, 67 degrees of freedom")
ax.set_title("The t-distribution: the normal curve with heavier tails")
ax.set_xlabel("standard errors from the centre")
ax.legend()
plt.show()

print(f"normal cut-off : {stats.norm.ppf(0.975):.3f}")
print(f"t cut-off, df=67: {stats.t.ppf(0.975, df=n - 1):.3f}")

With 67 degrees of freedom the t-distribution is almost the normal one: the cut-off is
1.996 instead of 1.96. With only 5 penguins it would be very different. The t version is
the one to use whenever the standard deviation comes from the sample, which is nearly
always, and scipy builds the whole interval in one call:

In [ ]:
# 95% confidence interval for the mean, from the t-distribution
ci_lower, ci_upper = stats.t.interval(0.95, df=n - 1, loc=mean_mass, scale=se_mass)
print(f"95% confidence interval for the Chinstrap average: {ci_lower:.0f} g to {ci_upper:.0f} g")

So the average Chinstrap body mass is plausibly anywhere between about 3,640 g and
3,826 g. Our 68 penguins cannot pin it down more tightly than that.

> **Exercise:**
>
> Build a 95% confidence interval for the average Chinstrap **flipper length**.

<details><summary>
Click here for a hint...
</summary>

Three numbers go into `stats.t.interval`: the degrees of freedom `n - 1`, the sample
average as `loc`, and the standard error `s / np.sqrt(n)` as `scale`.
</details>

In [ ]:
flipper = chinstrap["flipper_length_mm"]
se_flipper = flipper.std() / np.sqrt(len(flipper))
low, high = stats.t.interval(0.95, df=len(flipper) - 1, loc=flipper.mean(), scale=se_flipper)
print(f"Chinstrap flipper length: {low:.1f} mm to {high:.1f} mm")

## 5. What the 95% means

It is tempting to read the interval as *"there is a 95 per cent chance the true average is
between 3,640 g and 3,826 g"*. That is not quite right, and the difference matters for
testing.

The true average is a fixed number. It is either inside this interval or it is not; there
is no chance involved any more. The 95 per cent describes the **method**: build intervals
this way on sample after sample, and 95 per cent of them will catch the true mean. Our
interval is one of those, and we cannot tell whether it is one of the lucky 95 or the
unlucky 5.

The pretend population from section 2 lets us check that claim, because there we know the
true mean is 3,750 g.

In [ ]:
# 1,000 fresh samples of 68 from the pretend population (mu = 3750, sigma = 400)
samples = rng.normal(loc=mu, scale=sigma, size=(1000, 68))

# one 95% interval per sample
means = samples.mean(axis=1)
standard_errors = samples.std(axis=1, ddof=1) / np.sqrt(68)
lowers, uppers = stats.t.interval(0.95, df=67, loc=means, scale=standard_errors)

# did each interval catch the true mean?
caught = (lowers <= mu) & (mu <= uppers)
print(f"{caught.sum()} of 1,000 intervals contain the true mean ({caught.mean():.1%})")

In [ ]:
# draw the first 100 intervals, one per row; the misses in red
fig, ax = plt.subplots(figsize=(8, 7))
for row in range(100):
    colour = "steelblue" if caught[row] else "crimson"
    ax.plot([lowers[row], uppers[row]], [row, row], color=colour, linewidth=2)

ax.axvline(mu, color="black", linestyle="--", label=f"true mean, {mu} g")
ax.set_title("100 samples, 100 confidence intervals")
ax.set_xlabel("body mass (g)")
ax.set_ylabel("sample")
ax.legend(loc="upper right")
plt.show()

Most intervals cross the dashed line; a handful, in red, miss it completely. Every one of
them was built in exactly the same way, and nothing about a red interval looks wrong from
the inside. **That is the 5 per cent**, and it never goes away: it is the price of
answering in a range rather than waiting for the whole population.

The level is a choice, and it is a trade. Ask for 90 per cent and the intervals come out
narrower, but about one in ten misses; ask for 99 per cent and they miss only one time in a
hundred, at the cost of being wider. 95 per cent is the usual compromise.

## 6. From an interval to a decision

A field guide states that Chinstrap penguins weigh **3,700 g** on average. Our sample
averages 3,733 g. Is the field guide wrong?

The interval answers it. Any value inside the 95 per cent interval is a plausible true
average, one our data cannot rule out. Any value outside it is one the data speaks against.

In [ ]:
for claim in [3700, 4000]:
    inside = ci_lower <= claim <= ci_upper
    verdict = "plausible" if inside else "not plausible"
    print(f"claimed average {claim} g: inside {ci_lower:.0f} to {ci_upper:.0f}? "
          f"{inside} -> {verdict}")

3,700 g sits comfortably inside the interval, so the 33 g gap between the field guide and
our sample is well within the wobble of an average of 68. A claim of 4,000 g would be a
different story.

You have just made a yes-or-no decision about a claim, using data, with a known 5 per cent
chance of being fooled. **That is a hypothesis test in all but name.** Notebook 03 makes it
formal, and shows it gives exactly the same answer.

> **Exercise:**
>
> No code needed. A public health team measured the daily salt intake of 200 adults in a
> city, and reports a 95 per cent confidence interval for the average of **2,100 mg to
> 2,500 mg**. The recommended limit is **2,300 mg**.
>
> 1. Can the team say that the city's adults are, on average, within the limit?
> 2. Can they say that the average is over the limit?
> 3. Write one sentence for the city council that is both honest and useful.

Your answers here:

<details><summary>
Click here to compare your answers...
</summary>

1. **No.** Plausible averages run up to 2,500 mg, over the limit.
2. **No, not that either.** Plausible averages run down to 2,100 mg, under it. The limit
   of 2,300 mg is inside the interval, so the data cannot tell these two apart.
3. For example: *"Our best estimate is that adults here average 2,100 to 2,500 mg of salt a
   day, so the average could be on either side of the 2,300 mg limit; a larger study would
   be needed to say which."*
</details>

## Wrapping up

In this notebook you:

- separated what you know, the sample statistics, from what you want to know, the
  population parameters.
- watched sample averages wobble around the true mean, and measured the wobble: the
  standard error, $s / \sqrt{n}$.
- saw averages come out bell-shaped even from lopsided data, the central limit theorem.
- built a 95% confidence interval with the t-distribution, and checked by simulation what
  the 95% really promises.
- used the interval to judge a claim: 3,700 g plausible, 4,000 g not.

Section 6 decided the field guide's claim by looking at an interval. In
`03_how_a_test_works.ipynb` you ask the same question the other way round: *if the field
guide were right, how surprising would our 68 penguins be?* The answer is a p-value.

## References & Further Reading

- [**scipy.stats.t**](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html):
  The t-distribution, including the `interval` method used in section 4.
- [**Seeing Theory: Frequentist Inference**](https://seeing-theory.brown.edu/frequentist-inference/index.html):
  An animated version of the simulation in section 5, with the confidence level on a
  slider.
- [**ModernDive: Confidence Intervals**](https://moderndive.com/v2/confidence-intervals.html):
  The same ideas built from resampling, a good second explanation.
- [**StatQuest: Confidence Intervals**](https://www.youtube.com/watch?v=TqOeMYtOc1w):
  The idea in under ten minutes.